# Suite STO — Stores

A store is the root object of a body of data: it holds schemas by name and the instances of its singleton schemas, and
its data is what those reach; everything that looks a schema up takes one. `Proxies.OfStore` and `Bindings.OfStore`
implement the protocol `Stores.Store`. Stores are isolated from one another; objects move between them as
snapshots.

In [ ]:
from types import SimpleNamespace

from mbse.Schemas.Framework import Bindings as Bd, JSON, Modules, Proxies, Reflection, Schemas as S, Stores, Validators
from support import conformance_problems, instance_problems, raises, text

Item = S.OfObject.Builder().name("Item").ref().properties(text("name")).create()
Holds = S.OfRelation.Builder().name("Holds").links("owner", "item").properties(text("label")).create()
Shelved = S.OfRelation.Builder().name("Shelved").links("shelf", "box").create()
Box = S.OfObject.Builder().name("Box").ref().properties(text("name")).relations(lambda r: r.name("items").of(Holds).me("owner"),
                                                                    lambda r: r.name("shelves").of(Shelved).me("box")).create()
Shelf = S.OfObject.Builder().name("Shelf").ref().singleton("the.Shelf").properties(text("name")).relations(
    lambda r: r.name("boxes").of(Shelved).me("shelf")).create()
Annex = S.OfObject.Builder().name("Annex").ref().singleton("the.Annex").relations(lambda r: r.name("boxes").of(Shelved).me("shelf")).create()
Item = S.OfObject.Builder(Item).relations(lambda r: r.name("boxes").of(Holds).me("item")).update()
Port = S.OfObject.Builder().properties(text("name")).relations(lambda r: r.name("wired").of(Holds).me("owner")).create()
Card = S.OfObject.Builder().name("Card").ref().properties(lambda p: p.name("port").of(Port)).create()


def stocked(*names):
    """A proxy store with the schemas named, of Item, Box, Holds, Shelf, Shelved, Annex and Card."""
    store = Proxies.OfStore()
    for name in names:
        store.register({s.name: s for s in (Item, Box, Holds, Shelf, Shelved, Annex, Card)}[name])
    return store

## STO-01 · A store holds schemas by name

In [ ]:
store = stocked("Item", "Box", "Holds")
assert store.names() == ("Schemas.Module", "Item", "Box", "Holds")  # every proxy store starts with the meta-schemas
assert store.schema("Box") is Box and store.registered("Holds") is Holds and store.name_of(Item) == "Item"
with raises(ValueError, match="schema 'Box' is already registered"):
    store.register(Box)
with raises(ValueError, match="a schema needs a name to be registered; name it with its builder's .name()"):
    store.register(S.OfObject.Builder().ref().create())
with raises(AttributeError, match="no schema registered as 'Nope'"):
    store.schema("Nope")
with raises(TypeError, match="'Holds' is a relation; no relation builder is exposed"):
    store.schema("Holds")
with raises(LookupError, match="no schema registered as 'Nope'"):
    store.registered("Nope")
with raises(LookupError, match="schema is not registered"):
    store.name_of(Port)
store.register(S.OfObject.Builder(Item).name("register").clone())  # a name a method shadows: through builder()
assert store.builder("register").name("r").create().schema_name() == "register"
with raises(AttributeError, match="_hidden"):
    getattr(store, "_hidden")
assert store.Box().name("b").create().name == "b" and store.builder("Box").name("c").create().name == "c"

## STO-02 · A store's data is what its singletons reach; the rest is transient

In [ ]:
import gc
import weakref

STOCK = ("Item", "Box", "Holds", "Shelf", "Shelved", "Card")
store = stocked(*STOCK)
shelf = store.singleton("the.Shelf")  # created with its schema
assert store.extent("Shelf") == (shelf,) and store.extent("Box") == ()
a = store.Item().name("a").create()
box = store.Box().name("box").items(lambda e: e.item(a).label("first")).create()
assert store.extent("Box") == () and store.extent("Item") == ()  # nothing reachable from a singleton links them
store.Shelf(shelf).boxes(lambda e: e.box(box)).update()
assert store.extent("Box") == (box,) and store.extent("Item") == (a,)  # reachable now, transitively
store.Box(box).items(lambda e: e.item(lambda i: i.name("inline"))).update()
assert [i.name for i in store.extent("Item")] == ["a", "inline"]
store.register(Annex)  # a second root, reaching the same box
store.Annex(store.singleton("the.Annex")).boxes(lambda e: e.box(box)).update()
assert store.extent("Box") == (box,) and store.extent("Annex") == (store.singleton("the.Annex"),)
card = store.Card().port(lambda p: p.name("p1")).create()
assert store.extent("Card") == ()

# What no singleton reaches is the program's to keep: the store lets it go.
loose = store.Box().name("loose").items(lambda e: e.item(store.Item().name("i").create())).create()
gone = weakref.ref(loose)
del loose
gc.collect()
assert gone() is None

# There is one instance of a singleton schema per store; decoding one updates it.
with raises(ValueError, match="create() would make a second 'Shelf'; its one instance is store.singleton('the.Shelf')"):
    store.Shelf().create()
with raises(ValueError, match="clone() would make a second 'Shelf'"):
    store.Shelf(shelf).clone()
with raises(ValueError, match="singleton 'the.Shelf' is already registered"):
    store.register(S.OfObject.Builder(Shelf).name("Shelf2").clone())
with raises(LookupError, match="no singleton named 'nope'"):
    store.singleton("nope")
other = stocked(*STOCK, "Annex")
copy = JSON.FromJSON(other).Reachable(Shelf, JSON.ToJSON(store).Reachable(Shelf, shelf))
assert copy is other.singleton("the.Shelf") and [b.name for b in other.extent("Box")] == ["box"]
assert other.extent("Annex") == (other.singleton("the.Annex"),)  # the annex in the snapshot updated other's own
with raises(AttributeError, match="no schema registered as 'Nope'"):
    store.extent("Nope")
with raises(TypeError, match="'Holds' is a relation"):
    store.extent("Holds")

## STO-03 · Stores are isolated; objects move between them as snapshots

In [ ]:
one, two = stocked("Item", "Box", "Holds"), stocked("Item", "Box", "Holds")
Other = S.OfObject.Builder().name("Box").ref().properties(text("title")).create()
three = Proxies.OfStore()
three.register(Other)  # the same name, another schema
assert three.Box().title("t").create().title == "t"
x = one.Item().name("x").create()
y = two.Item().name("y").create()
with raises(TypeError, match="the object belongs to another store"):
    one.Box().items(lambda e: e.item(y))
with raises(TypeError, match="the object belongs to another store"):
    two.Item(x)
b = one.Box().name("b").items(lambda e: e.item(x).label("l")).create()
assert Proxies.store_of(b) is one and Proxies.store_of(b) is not two
moved = JSON.FromJSON(two).Reachable(Box, JSON.ToJSON(one).Reachable(Box, b))
assert Proxies.store_of(moved) is two and moved is not b and moved.name == "b"
assert Validators.Validate(two).Reachable(Box, moved) == []
assert Validators.Validate(three)(Box, b) == ["the value is a 'Box', not an instance of the given schema"]
card = stocked("Card").Card().port(lambda p: p.name("p")).create()
assert Proxies.store_of(card.port) is Proxies.store_of(card)  # a value object's store is its owner's
with raises(TypeError, match="the value does not belong to a store"):
    Proxies.store_of("text")
module = Modules.module(one, [Item])
assert Proxies.store_of(module) is one

## STO-04 · Every implementation meets the protocol

In [ ]:
assert conformance_problems(Proxies.OfStore, Stores.Store) == []
assert conformance_problems(Bd.OfStore, Stores.Store) == []
assert instance_problems(Proxies.OfStore(), Stores.Store) == []


class Thing:
    """A class of the program's own, bound to Item or to a singleton schema, One."""

    def __init__(self, binding, name=None):
        self.binding, self.name = binding, name

    def identity(self):
        return id(self)

    def schema_name(self):
        return "One" if self.binding is ONE_BINDING else "Item"

    def owner(self):
        return None

    def accept(self, visitor):
        Bd.accept(self.binding, self, visitor)


THING = Bd.Binding(Item, lambda t: Bd.State({"name": t.name}), lambda s: Thing(THING, s.values.get("name")),
                   lambda t, s: (setattr(t, "name", s.values.get("name")), t)[1], implied=["boxes"])
bound = Bd.OfStore([(Item, lambda instance=None: Bd.Builder(THING, instance))], [Holds])
assert instance_problems(bound, Stores.Store) == [] and bound.extent("Item") == ()
assert bound.names() == ("Item", "Holds")  # a store of bound classes holds no meta-schemas
One = S.OfObject.Builder().name("One").ref().singleton("the.One").properties(text("name")).create()
ONE_BINDING = Bd.Binding(One, lambda t: Bd.State({"name": t.name}), lambda s: Thing(ONE_BINDING, s.values.get("name")))
ones = Bd.OfStore([(One, lambda instance=None: Bd.Builder(ONE_BINDING, instance))])
assert isinstance(ones.singleton("the.One"), Thing) and ones.extent("One") == (ones.singleton("the.One"),)
first = bound.Item().property("name", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("first")))).create()
assert bound.builder("Item", first).clone() is not first and bound.builder("Item", first).update() is first
assert bound.member(first, "name") == "first" and bound.extent("Item") == ()  # no singleton reaches it
assert JSON.FromJSON(bound)(Item, JSON.ToJSON(bound)(Item, first)).name == "first"
with raises(AttributeError, match="Nope"):
    bound.Nope
with raises(TypeError, match="'Holds' is a relation"):
    bound.builder("Holds")
with raises(TypeError, match="'Holds' is a relation"):
    bound.extent("Holds")

## STO-05 · A value object not yet placed belongs to no store

In [ ]:
loose = Proxies._RecordBuilder(Port).name("loose").build()  # built, but not placed in an owner
seen = []
loose.accept(type("Recorder", (), {"identify": lambda self, v: None,
                                   "property": lambda self, n, cb: seen.append(n),
                                   "adjacency": lambda self, n, cb: seen.append(n)})())
assert seen == ["name"] and loose.schema_name() == ""  # its properties, and no entries; its schema is named nowhere
with raises(TypeError, match="the value does not belong to a store"):
    Proxies.store_of(loose)
with raises(TypeError, match="an object is created through a link only by a store's builder"):
    Proxies._RecordBuilder(Port).wired(lambda e: e.item(lambda i: i.name("new")))

## STO-06 · Stores combined: each name its store's, the singletons all of theirs, extents reaching across them

In [ ]:
Points = S.OfRelation.Builder().name("Points").links("index", "target").create()
IndexSchema = S.OfObject.Builder().name("Index").ref().singleton("Index").relations(
    lambda r: r.name("targets").of(Points).me("index")).create()


class Index:
    """A root of the program's own, which links to schemas held by another store."""

    def __init__(self, targets=()):
        self.targets = list(targets)

    def identity(self):
        return id(self)

    def schema_name(self):
        return "Index"

    def owner(self):
        return None

    def accept(self, visitor):
        Bd.accept(INDEXED, self, visitor)


INDEXED = Bd.Binding(IndexSchema, lambda index: Bd.State({}, {"targets": [Bd.Entry({"target": t}) for t in index.targets]}),
                     lambda state: Index(e.links["target"] for e in state.entries.get("targets", [])))
Phone = S.OfObject.Builder().name("Phone").properties(text("number")).create()
Card = S.OfObject.Builder().name("Card").ref().properties(text("name")).create()
schemas = Reflection.store([Phone])
indexes = Bd.OfStore([(IndexSchema, lambda instance=None: Bd.Builder(INDEXED, instance))], [Points])
indexes.singleton("Index").targets = [Card, Phone]  # a schema the other store does not list, and one it does
both = Stores.Combined(schemas, indexes)
assert both.names() == (*schemas.names(), "Index", "Points") and both.stores == (schemas, indexes)
assert list(both.extent("Schemas.Object")) == [Phone, Card]  # reached from either store's singletons
assert list(both.extent("Index")) == [indexes.singleton("Index")] and both.singleton("Index") is indexes.singleton("Index")
assert both.singleton("Schemas.Catalog") is schemas.singleton("Schemas.Catalog")
assert both.schema("Index") is IndexSchema and both.registered("Points") is Points and both.name_of(Points) == "Points"
assert both.member(Card, "name") == "Card" and isinstance(both.builder("Schemas.Property"), Bd.Builder)
with raises(AttributeError, match="no schema registered as 'Nope'"):
    both.schema("Nope")
with raises(LookupError, match="no schema registered as 'Nope'"):
    both.registered("Nope")
with raises(LookupError, match="schema is not registered"):
    both.name_of(Phone)
with raises(LookupError, match="no singleton named 'Nope'"):
    both.singleton("Nope")
with raises(ValueError, match="schema 'Schemas.Catalog' is registered by two of the stores"):
    Stores.Combined(schemas, Reflection.store())
nested = Stores.Combined(Stores.Combined(schemas), indexes, SimpleNamespace(names=lambda: ()))  # and one with no singletons
assert list(nested.extent("Schemas.Object")) == [Phone, Card] and nested.singleton("Index") is indexes.singleton("Index")